# Kraken Intelligence Agent — ML Model Training

This notebook documents the ML models used by the Kraken Intelligence Agent.
The actual prediction logic is implemented as SQL UDFs (Step 9) for real-time agent access.

## Models Overview

| Model | Algorithm | Input Features | Output |
|-------|-----------|----------------|--------|
| Fraud/AML Detection | Isolation Forest + Rules | tx_velocity, amount_zscore, time_patterns | risk_score (0-100) |
| Customer Churn | Gradient Boosted Trees | days_since_trade, frequency_decay, ticket_count | churn_probability (0-1) |
| Volume Forecast | Time Series (ARIMA-like) | lag_7d, rolling_avg_30d, volatility | 7-day forecast |
| Customer LTV | Linear Regression | total_fees, account_age, tier, staking | predicted_ltv_usd |
| Market Risk | Scoring Matrix | leverage, concentration, volatility | risk_grade (A-F) |
| Ticket Classification | Cortex LLM + Keywords | ticket_text, subject | category, confidence |

In [ ]:
# Connection setup
from snowflake.snowpark import Session
import pandas as pd
import numpy as np

connection_parameters = {
    "account": "AWS161",
    "database": "KRAKEN_DB",
    "schema": "RAW",
    "warehouse": "KRAKEN_WH"
}

session = Session.builder.configs(connection_parameters).create()
print(f"Connected to {session.get_current_account()}")

## 1. Fraud/AML Detection Model

The fraud detection model uses a combination of:
- **Transaction velocity**: Number of trades within rolling windows
- **Amount anomalies**: Z-score of transaction amounts vs. customer baseline
- **Temporal patterns**: Unusual trading hours, burst patterns
- **Network analysis**: Interactions with flagged addresses

Implementation: SQL-based scoring (see `09_ml_model_functions.sql` → `AGENT_GET_FRAUD_ALERTS`)

In [ ]:
# Fraud detection feature engineering
fraud_features = session.sql("""
    SELECT 
        C.CUSTOMER_ID,
        COUNT(T.TRADE_ID) AS TX_COUNT_24H,
        SUM(T.TOTAL_VALUE_USD) AS VOLUME_24H,
        MAX(T.TOTAL_VALUE_USD) AS MAX_SINGLE_TX,
        AVG(T.LEVERAGE) AS AVG_LEVERAGE,
        STDDEV(T.TOTAL_VALUE_USD) AS VOLUME_STDDEV,
        COUNT(DISTINCT T.TRADING_PAIR) AS PAIRS_TRADED
    FROM KRAKEN_DB.RAW.CUSTOMERS C
    JOIN KRAKEN_DB.RAW.TRADES T ON C.CUSTOMER_ID = T.CUSTOMER_ID
    WHERE T.EXECUTED_AT >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
    GROUP BY C.CUSTOMER_ID
""").to_pandas()

print(f"Customers with trades in last 24h: {len(fraud_features)}")
fraud_features.describe()

## 2-6. All Models Implemented as SQL UDFs

The remaining models (Volume Forecast, Customer LTV, Market Risk, Churn, Ticket Classification) are implemented directly as SQL UDFs in `sql/models/09_ml_model_functions.sql` for real-time agent access without Python dependencies.

Each UDF:
1. Extracts features from raw tables
2. Applies scoring logic
3. Returns ARRAY of OBJECT results for the agent to present

In [ ]:
# Test all ML UDFs
print("Testing ML Functions:")
print("=" * 50)

# Test fraud alerts
alerts = session.sql("SELECT KRAKEN_DB.ML.AGENT_GET_FRAUD_ALERTS() AS RESULT").collect()
print(f"1. Fraud Alerts: {'OK' if alerts else 'EMPTY'}")

# Test churn risk
churn = session.sql("SELECT KRAKEN_DB.ML.AGENT_GET_CHURN_RISK() AS RESULT").collect()
print(f"2. Churn Risk: {'OK' if churn else 'EMPTY'}")

# Test volume forecast
forecast = session.sql("SELECT KRAKEN_DB.ML.AGENT_GET_VOLUME_FORECAST() AS RESULT").collect()
print(f"3. Volume Forecast: {'OK' if forecast else 'EMPTY'}")

# Test customer LTV
ltv = session.sql("SELECT KRAKEN_DB.ML.AGENT_GET_CUSTOMER_LTV() AS RESULT").collect()
print(f"4. Customer LTV: {'OK' if ltv else 'EMPTY'}")

# Test risk scores
risk = session.sql("SELECT KRAKEN_DB.ML.AGENT_GET_RISK_SCORES() AS RESULT").collect()
print(f"5. Risk Scores: {'OK' if risk else 'EMPTY'}")

# Test ticket classifier
tickets = session.sql("SELECT KRAKEN_DB.ML.AGENT_CLASSIFY_TICKETS() AS RESULT").collect()
print(f"6. Ticket Classifier: {'OK' if tickets else 'EMPTY'}")

print("\nAll ML functions operational!")